In [ ]:
import torch
import matplotlib.pyplot as plt

from torchvision.transforms.v2.functional import to_pil_image
from torchvision.models import get_model, get_model_weights
from torchvision.io import decode_image

from torchcam.utils import overlay_mask
from torchcam.methods import LayerCAM

from pathlib import Path

In [ ]:
images_dir = Path(r"C:\Users\frolu\OneDrive\Skrivbord\skol kodning\Book")

weights = get_model_weights("resnet18").DEFAULT
model = get_model("resnet18", weights=weights).eval()

preprocess = weights.transforms()

cam_extractor = LayerCAM(model, target_layer="layer4")

In [ ]:
images = [
    ("apa.png", "apa", "positive"),
    ("Lemur.png", "apa", "negative"),

    ("nebbdjur.png", "näbbdjur", "positive"),
    ("sverd_haj.png", "näbbdjur", "negative"),
    
    ("Barr_tred.png", "barrträd", "positive"),
    ("palm.png", "barrträd", "negative"),

]

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
axes = axes.flatten()

for ax, (file, name, label_type) in zip(axes, images):
    img = decode_image(file)
    input_tensor = preprocess(img).unsqueeze(0)

    out = model(input_tensor)
    activation_map = cam_extractor(out.squeeze(0).argmax().item(), out)

    result = overlay_mask(
        to_pil_image(img),
        to_pil_image(activation_map[0].squeeze(0), mode='F'),
        alpha=0.5
    )

    ax.imshow(result)
    ax.set_title(f"{name} ({label_type})")
    ax.axis("off")

plt.tight_layout()
plt.show()